### Process HRRRAK Grib Files: Large Juneau Domain

Notebook contents 
* Making a copy of other process script for the large juneau domain to process WY2024 and WY2025 

created by Cassie Lumbrazo\
last updated: Feb 2026\
run location: UAS linux\
python environment: **rasterio**

In [1]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_theme()
# plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy

from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from matplotlib import ticker

import rioxarray
import rasterio 
import cfgrib
import os

In [2]:
pwd

'/home/cassie/python/repos/snow_model_forcing/large_juneau_domain'

### Open all the HRRR-AK Grib Files for a Water Year (WY) and Create a Simple NetCDF

To scale up the process for multiple HRRR .grib2 files across a folder structure, we need to:

* Walk through directories inside folderspath (`/hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2024`).
* Identify .grib2 files inside each date-named subfolder (`hrrr.20231001`, `hrrr.20231002`, etc.).
* Open and process all files for a given date just like in your original script.
* Merge all processed datasets and store them in a structured way.

* and deal with CRS issues


In [3]:
# adding this for parallel processing
from concurrent.futures import ProcessPoolExecutor

### Large Juneau Domain, f567, WY2025

In [5]:
import xarray as xr
import numpy as np
import pygrib
import os
from concurrent.futures import ProcessPoolExecutor
import rioxarray  # For CRS handling
import pandas as pd

# Define main folder path
folderspath = '/hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2025'

# List all date folders
date_folders = sorted([f for f in os.listdir(folderspath) if f.startswith("hrrr.")])

# List of desired variables (short names from GRIB output, keeping naming conventions)
desired_vars = [
    't',        # Temperature (surface level)
    'sp',       # Surface pressure
    '2t',       # 2 metre temperature
    '2r',       # 2 metre relative humidity
    'tp',       # Total Precipitation
    'prate',    # Precipitation rate
    '10u',      # 10 metre U wind component
    '10v',      # 10 metre V wind component
    'sdswrf',   # Surface downward short-wave radiation flux
    'sdlwrf',   # Surface downward long-wave radiation flux
    # Extras
    'gust',     # Wind speed (gust)
    'tcc',      # Total Cloud Cover
    'lcc',      # Low cloud cover
    'mcc',      # Medium cloud cover
    'hcc',      # High cloud cover
    'lai',      # Leaf Area Index
    '2d',       # 2 metre dewpoint temperature
    '2sh',      # 2 metre specific humidity
    'suswrf',   # Surface upward short-wave radiation flux
    'sulwrf',   # Surface upward long-wave radiation flux
    'orog',     # Orography
    'sdwe',     # Water equivalent of accumulated snow depth
    'sde',      # Snow depth
    'veg',      # Vegetation
    'vgtyp',    # Vegetation Type
    'gflux',    # Ground heat flux
]

def process_grib_file(grib_path):
    """ Process a single .grib2 file and return an xarray dataset """
    try:
        # Extract init time and forecast step from filename
        # Example: 'hrrr.t00z.wrfsfcf05.ak.grib2' -> init_hour=0, forecast_step=5
        filename = os.path.basename(grib_path)
        init_hour = int(filename.split('.')[1][1:3])  # 't00z' -> 0
        forecast_step = int(filename.split('.')[2][-2:])  # 'wrfsfcf05' -> '05' -> 5
        
        # Get date from folder name
        date_str = grib_path.split('/')[-2].split('.')[1]  # e.g., '20241001'
        init_datetime = pd.to_datetime(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]} {init_hour:02d}:00")
        valid_time = init_datetime + pd.Timedelta(hours=forecast_step)
        
        # Collect desired vars using pygrib
        grbs = pygrib.open(grib_path)
        var_info = {}
        for grb in grbs:
            short_name = grb.shortName.lower()
            if short_name in desired_vars:
                # Special filter: for 't', only keep surface level
                if short_name == 't' and not (grb.typeOfLevel == 'surface'):
                    continue
                key = f"{short_name}_{grb.level}_{grb.typeOfLevel}"
                var_info[key] = {
                    'short_name': short_name,
                    'level': grb.level,
                    'typeOfLevel': grb.typeOfLevel,
                    'data': grb.values,
                    'lat': grb.latlons()[0],
                    'lon': grb.latlons()[1],
                    'attrs': {  # Preserve GRIB metadata
                        'long_name': getattr(grb, 'name', short_name),
                        'units': getattr(grb, 'units', ''),
                        'GRIB_shortName': getattr(grb, 'shortName', short_name),
                        'GRIB_name': getattr(grb, 'name', ''),
                        'GRIB_cfName': getattr(grb, 'cfName', ''),
                        'GRIB_cfVarName': getattr(grb, 'cfVarName', ''),
                        'level': grb.level,
                        'typeOfLevel': grb.typeOfLevel
                    }
                }
        grbs.close()
        
        if not var_info:
            print(f"⚠️ Skipping {grib_path}: No desired variables found")
            return None
        
        # Create merged dataset
        datasets = []
        first_key = list(var_info.keys())[0]
        lat = var_info[first_key]['lat']
        lon = var_info[first_key]['lon']
        
        grouped_vars = {}
        for key, info in var_info.items():
            group_key = f"{info['short_name']}_{info['typeOfLevel']}"
            if group_key not in grouped_vars:
                grouped_vars[group_key] = []
            grouped_vars[group_key].append(info)
        
        for group_key, infos in grouped_vars.items():
            short_name = infos[0]['short_name']
            type_level = infos[0]['typeOfLevel']
            attrs = infos[0]['attrs']  # Use attrs from first (assuming consistent)
            
            if len(infos) == 1:
                info = infos[0]
                data = info['data']
                level = info['level']
                
                if type_level == 'surface' or (type_level == 'heightAboveGround' and level == 0):
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                elif type_level == 'heightAboveGround' and level in [2, 10]:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs={**attrs, 'heightAboveGround': level},  # Add height as attr
                        name=short_name
                    )
                else:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                datasets.append(da)
            else:
                # Multi-level (rare now)
                levels = [info['level'] for info in infos]
                data_stack = np.stack([info['data'] for info in infos], axis=0)
                da = xr.DataArray(
                    data_stack,
                    dims=['level', 'y', 'x'],
                    coords={
                        'level': levels,
                        'y': range(data_stack.shape[1]),
                        'x': range(data_stack.shape[2]),
                        'latitude': (['y', 'x'], lat),
                        'longitude': (['y', 'x'], lon)
                    },
                    attrs=attrs,
                    name=short_name
                )
                datasets.append(da)
        
        ds_merged = xr.merge(datasets, compat='override')
        
        # Calculate wind speed and direction (from 10u and 10v), with attrs
        if '10u' in ds_merged and '10v' in ds_merged:
            ds_merged['wind'] = np.sqrt(ds_merged['10u']**2 + ds_merged['10v']**2)
            ds_merged['wind'].attrs.update({
                'long_name': '10 metre wind speed calculated from u and v wind components',
                'units': 'm s**-1',  # Assuming standard units
                'GRIB_shortName': '10m wind',
                'standard_name': 'wind speed',
                'GRIB_name': '10 metre wind speed',
                'GRIB_cfName': 'wind_speed',
                'GRIB_cfVarName': 'wind10'
            })
            
            # Wind direction: atan2(v, u) gives direction in degrees from east (positive x), convert to 0-360
            wind_dir_rad = np.arctan2(ds_merged['10v'], ds_merged['10u'])
            ds_merged['wind_dir'] = (wind_dir_rad * 180 / np.pi + 360) % 360
            ds_merged['wind_dir'].attrs.update({
                'long_name': '10 metre wind direction (from which wind blows)',
                'units': 'degrees',
                'GRIB_shortName': '10m wind dir',
                'standard_name': 'wind_from_direction',
                'GRIB_name': '10 metre wind direction',
                'GRIB_cfName': 'wind_from_direction',
                'GRIB_cfVarName': 'winddir10'
            })
        
        # Rename variables (as in old code) - attrs should be preserved
        ds_merged = ds_merged.rename({
            't': 'temp_surface',
            'sp': 'pressure',
            '2t': 'temp',
            '2r': 'rh',
            'sdwe': 'swe',
            'sde': 'snowdepth',
            'sdswrf': 'swrad',
            'sdlwrf': 'lwrad',
            'tp': 'precip_total',
            'prate': 'precip_rate'
        })
        
        # Drop unnecessary variables if any
        drop_vars = [var for var in ['unknown'] if var in ds_merged]
        ds_merged = ds_merged.drop_vars(drop_vars, errors='ignore')
        
        # Assign time coordinates (mimic old code: time as valid_time, add step)
        step = pd.Timedelta(hours=forecast_step)
        ds_merged = ds_merged.assign_coords({
            'time': valid_time,
            'step': step,
            'valid_time': valid_time
        })
        
        return ds_merged
    
    except Exception as e:
        print(f"⚠️ Error processing {grib_path}: {e}")
        return None

# Function to process a date folder in parallel
def process_date_folder(date_folder):
    date_path = os.path.join(folderspath, date_folder)
    grib_files = sorted([os.path.join(date_path, f) for f in os.listdir(date_path) if f.endswith(".grib2")])
    
    with ProcessPoolExecutor() as executor:
        ds_list = list(filter(None, executor.map(process_grib_file, grib_files)))
    
    if ds_list:
        try:
            return xr.concat(ds_list, dim="time")
        except ValueError as e:
            print(f"⚠️ Skipping {date_folder} due to concatenation error: {e}")
            return None
    return None

# Process all date folders in parallel
with ProcessPoolExecutor() as executor:
    all_ds_list = list(filter(None, executor.map(process_date_folder, date_folders)))

# Merge all datasets across all days
if all_ds_list:
    final_ds = xr.concat(all_ds_list, dim="time")
    print("✅ Successfully merged all datasets!")
    
    # --- Save raw NetCDF before CRS processing ---
    raw_output_path = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2025_raw.nc"
    final_ds.to_netcdf(raw_output_path)
    print(f"📁 Saved raw dataset to {raw_output_path}")
    
    # --- CRS Check and Reprojection ---
    print(f"Original CRS: {final_ds.rio.crs}")
    
    # Assign CRS if not set (assuming WGS84 lat/lon)
    final_ds.rio.write_crs("EPSG:4326", inplace=True)
    print("Assigned CRS: EPSG:4326")
    
    # Drop old integer x/y coords to avoid rename conflict
    final_ds = final_ds.drop_vars(['x', 'y'], errors='ignore')
    
    # Rename dimensions and set spatial dims
    final_ds = final_ds.rename({'longitude': 'x', 'latitude': 'y'})
    final_ds = final_ds.rio.set_spatial_dims(x_dim='x', y_dim='y')
    
    # Extract 1D x and y coordinates from the 2D grid
    x_1d = final_ds['x'].isel(y=0)  # same x values across rows
    y_1d = final_ds['y'].isel(x=0)  # same y values down columns
    
    # Assign 1D coords
    final_ds = final_ds.assign_coords(x=x_1d, y=y_1d)
    
    # Drop old 2D coords
    final_ds = final_ds.drop_vars(['x', 'y'])
    final_ds = final_ds.set_coords([])
    final_ds = final_ds.assign_coords({'x': x_1d, 'y': y_1d})
    
    # Reproject to UTM
    ds_utm = final_ds.rio.reproject("EPSG:32608")
    print("Reprojected to UTM (EPSG:32608)")
    
    # Save the reprojected dataset
    utm_output_path = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2025_utm.nc"
    ds_utm.to_netcdf(utm_output_path)
    print(f"📁 Saved reprojected dataset to {utm_output_path}")
else:
    print("⚠️ No valid datasets found.")

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2025_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2025_utm.nc


### Large Juneau Domain, f567, WY2024

In [5]:
import xarray as xr
import numpy as np
import pygrib
import os
from concurrent.futures import ProcessPoolExecutor
import rioxarray  # For CRS handling
import pandas as pd

# Define main folder path
folderspath = '/hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2024'

# List all date folders
date_folders = sorted([f for f in os.listdir(folderspath) if f.startswith("hrrr.")])

# List of desired variables (short names from GRIB output, keeping naming conventions)
desired_vars = [
    't',        # Temperature (surface level)
    'sp',       # Surface pressure
    '2t',       # 2 metre temperature
    '2r',       # 2 metre relative humidity
    'tp',       # Total Precipitation
    'prate',    # Precipitation rate
    '10u',      # 10 metre U wind component
    '10v',      # 10 metre V wind component
    'sdswrf',   # Surface downward short-wave radiation flux
    'sdlwrf',   # Surface downward long-wave radiation flux
    # Extras
    'gust',     # Wind speed (gust)
    'tcc',      # Total Cloud Cover
    'lcc',      # Low cloud cover
    'mcc',      # Medium cloud cover
    'hcc',      # High cloud cover
    'lai',      # Leaf Area Index
    '2d',       # 2 metre dewpoint temperature
    '2sh',      # 2 metre specific humidity
    'suswrf',   # Surface upward short-wave radiation flux
    'sulwrf',   # Surface upward long-wave radiation flux
    'orog',     # Orography
    'sdwe',     # Water equivalent of accumulated snow depth
    'sde',      # Snow depth
    'veg',      # Vegetation
    'vgtyp',    # Vegetation Type
    'gflux',    # Ground heat flux
]

def process_grib_file(grib_path):
    """ Process a single .grib2 file and return an xarray dataset """
    try:
        # Extract init time and forecast step from filename
        # Example: 'hrrr.t00z.wrfsfcf05.ak.grib2' -> init_hour=0, forecast_step=5
        filename = os.path.basename(grib_path)
        init_hour = int(filename.split('.')[1][1:3])  # 't00z' -> 0
        forecast_step = int(filename.split('.')[2][-2:])  # 'wrfsfcf05' -> '05' -> 5
        
        # Get date from folder name
        date_str = grib_path.split('/')[-2].split('.')[1]  # e.g., '20241001'
        init_datetime = pd.to_datetime(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]} {init_hour:02d}:00")
        valid_time = init_datetime + pd.Timedelta(hours=forecast_step)
        
        # Collect desired vars using pygrib
        grbs = pygrib.open(grib_path)
        var_info = {}
        for grb in grbs:
            short_name = grb.shortName.lower()
            if short_name in desired_vars:
                # Special filter: for 't', only keep surface level
                if short_name == 't' and not (grb.typeOfLevel == 'surface'):
                    continue
                key = f"{short_name}_{grb.level}_{grb.typeOfLevel}"
                var_info[key] = {
                    'short_name': short_name,
                    'level': grb.level,
                    'typeOfLevel': grb.typeOfLevel,
                    'data': grb.values,
                    'lat': grb.latlons()[0],
                    'lon': grb.latlons()[1],
                    'attrs': {  # Preserve GRIB metadata
                        'long_name': getattr(grb, 'name', short_name),
                        'units': getattr(grb, 'units', ''),
                        'GRIB_shortName': getattr(grb, 'shortName', short_name),
                        'GRIB_name': getattr(grb, 'name', ''),
                        'GRIB_cfName': getattr(grb, 'cfName', ''),
                        'GRIB_cfVarName': getattr(grb, 'cfVarName', ''),
                        'level': grb.level,
                        'typeOfLevel': grb.typeOfLevel
                    }
                }
        grbs.close()
        
        if not var_info:
            print(f"⚠️ Skipping {grib_path}: No desired variables found")
            return None
        
        # Create merged dataset
        datasets = []
        first_key = list(var_info.keys())[0]
        lat = var_info[first_key]['lat']
        lon = var_info[first_key]['lon']
        
        grouped_vars = {}
        for key, info in var_info.items():
            group_key = f"{info['short_name']}_{info['typeOfLevel']}"
            if group_key not in grouped_vars:
                grouped_vars[group_key] = []
            grouped_vars[group_key].append(info)
        
        for group_key, infos in grouped_vars.items():
            short_name = infos[0]['short_name']
            type_level = infos[0]['typeOfLevel']
            attrs = infos[0]['attrs']  # Use attrs from first (assuming consistent)
            
            if len(infos) == 1:
                info = infos[0]
                data = info['data']
                level = info['level']
                
                if type_level == 'surface' or (type_level == 'heightAboveGround' and level == 0):
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                elif type_level == 'heightAboveGround' and level in [2, 10]:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs={**attrs, 'heightAboveGround': level},  # Add height as attr
                        name=short_name
                    )
                else:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                datasets.append(da)
            else:
                # Multi-level (rare now)
                levels = [info['level'] for info in infos]
                data_stack = np.stack([info['data'] for info in infos], axis=0)
                da = xr.DataArray(
                    data_stack,
                    dims=['level', 'y', 'x'],
                    coords={
                        'level': levels,
                        'y': range(data_stack.shape[1]),
                        'x': range(data_stack.shape[2]),
                        'latitude': (['y', 'x'], lat),
                        'longitude': (['y', 'x'], lon)
                    },
                    attrs=attrs,
                    name=short_name
                )
                datasets.append(da)
        
        ds_merged = xr.merge(datasets, compat='override')
        
        # Calculate wind speed and direction (from 10u and 10v), with attrs
        if '10u' in ds_merged and '10v' in ds_merged:
            ds_merged['wind'] = np.sqrt(ds_merged['10u']**2 + ds_merged['10v']**2)
            ds_merged['wind'].attrs.update({
                'long_name': '10 metre wind speed calculated from u and v wind components',
                'units': 'm s**-1',  # Assuming standard units
                'GRIB_shortName': '10m wind',
                'standard_name': 'wind speed',
                'GRIB_name': '10 metre wind speed',
                'GRIB_cfName': 'wind_speed',
                'GRIB_cfVarName': 'wind10'
            })
            
            # Wind direction: atan2(v, u) gives direction in degrees from east (positive x), convert to 0-360
            wind_dir_rad = np.arctan2(ds_merged['10v'], ds_merged['10u'])
            ds_merged['wind_dir'] = (wind_dir_rad * 180 / np.pi + 360) % 360
            ds_merged['wind_dir'].attrs.update({
                'long_name': '10 metre wind direction (from which wind blows)',
                'units': 'degrees',
                'GRIB_shortName': '10m wind dir',
                'standard_name': 'wind_from_direction',
                'GRIB_name': '10 metre wind direction',
                'GRIB_cfName': 'wind_from_direction',
                'GRIB_cfVarName': 'winddir10'
            })
        
        # Rename variables (as in old code) - attrs should be preserved
        ds_merged = ds_merged.rename({
            't': 'temp_surface',
            'sp': 'pressure',
            '2t': 'temp',
            '2r': 'rh',
            'sdwe': 'swe',
            'sde': 'snowdepth',
            'sdswrf': 'swrad',
            'sdlwrf': 'lwrad',
            'tp': 'precip_total',
            'prate': 'precip_rate'
        })
        
        # Drop unnecessary variables if any
        drop_vars = [var for var in ['unknown'] if var in ds_merged]
        ds_merged = ds_merged.drop_vars(drop_vars, errors='ignore')
        
        # Assign time coordinates (mimic old code: time as valid_time, add step)
        step = pd.Timedelta(hours=forecast_step)
        ds_merged = ds_merged.assign_coords({
            'time': valid_time,
            'step': step,
            'valid_time': valid_time
        })
        
        return ds_merged
    
    except Exception as e:
        print(f"⚠️ Error processing {grib_path}: {e}")
        return None

# Function to process a date folder in parallel
def process_date_folder(date_folder):
    date_path = os.path.join(folderspath, date_folder)
    grib_files = sorted([os.path.join(date_path, f) for f in os.listdir(date_path) if f.endswith(".grib2")])
    
    with ProcessPoolExecutor() as executor:
        ds_list = list(filter(None, executor.map(process_grib_file, grib_files)))
    
    if ds_list:
        try:
            return xr.concat(ds_list, dim="time")
        except ValueError as e:
            print(f"⚠️ Skipping {date_folder} due to concatenation error: {e}")
            return None
    return None

# Process all date folders in parallel
with ProcessPoolExecutor() as executor:
    all_ds_list = list(filter(None, executor.map(process_date_folder, date_folders)))

# Merge all datasets across all days
if all_ds_list:
    final_ds = xr.concat(all_ds_list, dim="time")
    print("✅ Successfully merged all datasets!")
    
    # --- Save raw NetCDF before CRS processing ---
    raw_output_path = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2024_raw.nc"
    final_ds.to_netcdf(raw_output_path)
    print(f"📁 Saved raw dataset to {raw_output_path}")
    
    # --- CRS Check and Reprojection ---
    print(f"Original CRS: {final_ds.rio.crs}")
    
    # Assign CRS if not set (assuming WGS84 lat/lon)
    final_ds.rio.write_crs("EPSG:4326", inplace=True)
    print("Assigned CRS: EPSG:4326")
    
    # Drop old integer x/y coords to avoid rename conflict
    final_ds = final_ds.drop_vars(['x', 'y'], errors='ignore')
    
    # Rename dimensions and set spatial dims
    final_ds = final_ds.rename({'longitude': 'x', 'latitude': 'y'})
    final_ds = final_ds.rio.set_spatial_dims(x_dim='x', y_dim='y')
    
    # Extract 1D x and y coordinates from the 2D grid
    x_1d = final_ds['x'].isel(y=0)  # same x values across rows
    y_1d = final_ds['y'].isel(x=0)  # same y values down columns
    
    # Assign 1D coords
    final_ds = final_ds.assign_coords(x=x_1d, y=y_1d)
    
    # Drop old 2D coords
    final_ds = final_ds.drop_vars(['x', 'y'])
    final_ds = final_ds.set_coords([])
    final_ds = final_ds.assign_coords({'x': x_1d, 'y': y_1d})
    
    # Reproject to UTM
    ds_utm = final_ds.rio.reproject("EPSG:32608")
    print("Reprojected to UTM (EPSG:32608)")
    
    # Save the reprojected dataset
    utm_output_path = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2024_utm.nc"
    ds_utm.to_netcdf(utm_output_path)
    print(f"📁 Saved reprojected dataset to {utm_output_path}")
else:
    print("⚠️ No valid datasets found.")

⚠️ Skipping /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2024/hrrr.20240522/hrrr.t21z.wrfsfcf06.ak.grib2: No desired variables found
✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2024_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2024_utm.nc


### ⚠️ Skipping /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2024/hrrr.20240522/hrrr.t21z.wrfsfcf06.ak.grib2: No desired variables found

So, there is a single datetime missing from that missing forecast.\
I feel like it's easier to correct in the netcdf, and just fill the timeseries with interpolation...

let's leave it for now, but keep it in mind for the future. 

Adding in WY2023...

### Large Juneau Domain, f567, WY2023

Something failed with the variable names...

In [1]:
import pygrib
import pandas as pd

grib_file = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023/hrrr.20221001/hrrr.t00z.wrfsfcf05.ak.grib2"

# Inspect a single GRIB file and summarize variable structure
rows = []
with pygrib.open(grib_file) as grbs:
    print(f"Inspecting: {grib_file}")
    print(f"Total GRIB messages: {grbs.messages}")

    for idx, grb in enumerate(grbs, start=1):
        rows.append(
            {
                "msg": idx,
                "name": grb.name,
                "shortName": getattr(grb, "shortName", ""),
                "typeOfLevel": getattr(grb, "typeOfLevel", ""),
                "level": getattr(grb, "level", None),
                "units": getattr(grb, "units", ""),
                "stepType": getattr(grb, "stepType", ""),
            }
        )

df = pd.DataFrame(rows)

print("\nUnique variables (shortName, typeOfLevel, level):")
print(
    df[["shortName", "name", "typeOfLevel", "level", "units"]]
    .drop_duplicates()
    .sort_values(["shortName", "typeOfLevel", "level"])
    .to_string(index=False)
)

# Focused check for temperature / relative humidity at 2m and 10m
shortname_candidates = ["2t", "2r", "t", "r", "r2", "rh"]
focus = df[
    (
        df["shortName"].str.lower().isin(shortname_candidates)
        | df["name"].str.contains("temperature|relative humidity", case=False, na=False)
    )
    & (df["typeOfLevel"].isin(["heightAboveGround", "surface"]))
]

print("\nTemperature / RH candidates at surface or heightAboveGround:")
if focus.empty:
    print("No temperature/RH candidates found with current filters.")
else:
    print(
        focus[["msg", "shortName", "name", "typeOfLevel", "level", "units"]]
        .drop_duplicates()
        .sort_values(["shortName", "typeOfLevel", "level", "msg"])
        .to_string(index=False)
    )

# Exact 2 m and 10 m diagnostic
hgt = df[df["typeOfLevel"] == "heightAboveGround"].copy()
check_2m_10m = hgt[hgt["level"].isin([2, 10])]

print("\nheightAboveGround messages at 2 m and 10 m:")
if check_2m_10m.empty:
    print("No heightAboveGround messages found at 2 m or 10 m.")
else:
    print(
        check_2m_10m[["msg", "shortName", "name", "level", "units"]]
        .drop_duplicates()
        .sort_values(["level", "shortName", "msg"])
        .to_string(index=False)
    )

Inspecting: /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023/hrrr.20221001/hrrr.t00z.wrfsfcf05.ak.grib2
Total GRIB messages: 171

Unique variables (shortName, typeOfLevel, level):
shortName                                                    name                 typeOfLevel  level              units
      10u                               10 metre U wind component           heightAboveGround     10            m s**-1
      10v                               10 metre V wind component           heightAboveGround     10            m s**-1
       2d                            2 metre dewpoint temperature           heightAboveGround      2                  K
       2r                               2 metre relative humidity           heightAboveGround      2                  %
      2sh                               2 metre specific humidity           heightAboveGround      2          kg kg**-1
       2t                                     2 metre temperature           heightAboveGro

In [11]:
'gust' in df['shortName'].values

True

In [12]:
import xarray as xr
import numpy as np
import pygrib
import os
from concurrent.futures import ProcessPoolExecutor
import rioxarray  # For CRS handling
import pandas as pd

# Define main folder path
folderspath = '/hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023'

# List all date folders
date_folders = sorted([f for f in os.listdir(folderspath) if f.startswith("hrrr.")])

# List of desired variables (short names from GRIB output, keeping naming conventions)
desired_vars = [
    't',        # Temperature (surface level)
    'sp',       # Surface pressure
    '2t',       # 2 metre temperature
    '2r',       # 2 metre relative humidity
    'tp',       # Total Precipitation
    'prate',    # Precipitation rate
    '10u',      # 10 metre U wind component
    '10v',      # 10 metre V wind component
    'sdswrf',   # Surface downward short-wave radiation flux
    'sdlwrf',   # Surface downward long-wave radiation flux
    # Extras
    'gust',     # Wind speed (gust)
    'tcc',      # Total Cloud Cover
    'lcc',      # Low cloud cover
    'mcc',      # Medium cloud cover
    'hcc',      # High cloud cover
    'lai',      # Leaf Area Index
    '2d',       # 2 metre dewpoint temperature
    '2sh',      # 2 metre specific humidity
    'suswrf',   # Surface upward short-wave radiation flux
    'sulwrf',   # Surface upward long-wave radiation flux
    'orog',     # Orography
    'sdwe',     # Water equivalent of accumulated snow depth
    'sde',      # Snow depth
    'veg',      # Vegetation
    'vgtyp',    # Vegetation Type
    'gflux',    # Ground heat flux
]

def process_grib_file(grib_path):
    """ Process a single .grib2 file and return an xarray dataset """
    try:
        # Extract init time and forecast step from filename
        # Example: 'hrrr.t00z.wrfsfcf05.ak.grib2' -> init_hour=0, forecast_step=5
        filename = os.path.basename(grib_path)
        init_hour = int(filename.split('.')[1][1:3])  # 't00z' -> 0
        forecast_step = int(filename.split('.')[2][-2:])  # 'wrfsfcf05' -> '05' -> 5
        
        # Get date from folder name
        date_str = grib_path.split('/')[-2].split('.')[1]  # e.g., '20241001'
        init_datetime = pd.to_datetime(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]} {init_hour:02d}:00")
        valid_time = init_datetime + pd.Timedelta(hours=forecast_step)
        
        # Collect desired vars using pygrib
        grbs = pygrib.open(grib_path)
        var_info = {}
        for grb in grbs:
            short_name = grb.shortName.lower()
            if short_name in desired_vars:
                # Special filter: for 't', only keep surface level
                if short_name == 't' and not (grb.typeOfLevel == 'surface'):
                    continue
                key = f"{short_name}_{grb.level}_{grb.typeOfLevel}"
                var_info[key] = {
                    'short_name': short_name,
                    'level': grb.level,
                    'typeOfLevel': grb.typeOfLevel,
                    'data': grb.values,
                    'lat': grb.latlons()[0],
                    'lon': grb.latlons()[1],
                    'attrs': {  # Preserve GRIB metadata
                        'long_name': getattr(grb, 'name', short_name),
                        'units': getattr(grb, 'units', ''),
                        'GRIB_shortName': getattr(grb, 'shortName', short_name),
                        'GRIB_name': getattr(grb, 'name', ''),
                        'GRIB_cfName': getattr(grb, 'cfName', ''),
                        'GRIB_cfVarName': getattr(grb, 'cfVarName', ''),
                        'level': grb.level,
                        'typeOfLevel': grb.typeOfLevel
                    }
                }
        grbs.close()
        
        if not var_info:
            print(f"⚠️ Skipping {grib_path}: No desired variables found")
            return None
        
        # Create merged dataset
        datasets = []
        first_key = list(var_info.keys())[0]
        lat = var_info[first_key]['lat']
        lon = var_info[first_key]['lon']
        
        grouped_vars = {}
        for key, info in var_info.items():
            group_key = f"{info['short_name']}_{info['typeOfLevel']}"
            if group_key not in grouped_vars:
                grouped_vars[group_key] = []
            grouped_vars[group_key].append(info)
        
        for group_key, infos in grouped_vars.items():
            short_name = infos[0]['short_name']
            type_level = infos[0]['typeOfLevel']
            attrs = infos[0]['attrs']  # Use attrs from first (assuming consistent)
            
            if len(infos) == 1:
                info = infos[0]
                data = info['data']
                level = info['level']
                
                if type_level == 'surface' or (type_level == 'heightAboveGround' and level == 0):
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                elif type_level == 'heightAboveGround' and level in [2, 10]:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs={**attrs, 'heightAboveGround': level},  # Add height as attr
                        name=short_name
                    )
                else:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                datasets.append(da)
            else:
                # Multi-level (rare now)
                levels = [info['level'] for info in infos]
                data_stack = np.stack([info['data'] for info in infos], axis=0)
                da = xr.DataArray(
                    data_stack,
                    dims=['level', 'y', 'x'],
                    coords={
                        'level': levels,
                        'y': range(data_stack.shape[1]),
                        'x': range(data_stack.shape[2]),
                        'latitude': (['y', 'x'], lat),
                        'longitude': (['y', 'x'], lon)
                    },
                    attrs=attrs,
                    name=short_name
                )
                datasets.append(da)
        
        ds_merged = xr.merge(datasets, compat='override')
        
        # Calculate wind speed and direction (from 10u and 10v), with attrs
        if '10u' in ds_merged and '10v' in ds_merged:
            ds_merged['wind'] = np.sqrt(ds_merged['10u']**2 + ds_merged['10v']**2)
            ds_merged['wind'].attrs.update({
                'long_name': '10 metre wind speed calculated from u and v wind components',
                'units': 'm s**-1',  # Assuming standard units
                'GRIB_shortName': '10m wind',
                'standard_name': 'wind speed',
                'GRIB_name': '10 metre wind speed',
                'GRIB_cfName': 'wind_speed',
                'GRIB_cfVarName': 'wind10'
            })
            
            # Wind direction: atan2(v, u) gives direction in degrees from east (positive x), convert to 0-360
            wind_dir_rad = np.arctan2(ds_merged['10v'], ds_merged['10u'])
            ds_merged['wind_dir'] = (wind_dir_rad * 180 / np.pi + 360) % 360
            ds_merged['wind_dir'].attrs.update({
                'long_name': '10 metre wind direction (from which wind blows)',
                'units': 'degrees',
                'GRIB_shortName': '10m wind dir',
                'standard_name': 'wind_from_direction',
                'GRIB_name': '10 metre wind direction',
                'GRIB_cfName': 'wind_from_direction',
                'GRIB_cfVarName': 'winddir10'
            })
        
        # Rename variables (as in old code) - attrs should be preserved
        ds_merged = ds_merged.rename({
            't': 'temp_surface',
            'sp': 'pressure',
            '2t': 'temp',
            '2r': 'rh',
            'sdwe': 'swe',
            'sde': 'snowdepth',
            'sdswrf': 'swrad',
            'sdlwrf': 'lwrad',
            'tp': 'precip_total',
            'prate': 'precip_rate'
        })
        
        # Drop unnecessary variables if any
        drop_vars = [var for var in ['unknown'] if var in ds_merged]
        ds_merged = ds_merged.drop_vars(drop_vars, errors='ignore')
        
        # Assign time coordinates (mimic old code: time as valid_time, add step)
        step = pd.Timedelta(hours=forecast_step)
        ds_merged = ds_merged.assign_coords({
            'time': valid_time,
            'step': step,
            'valid_time': valid_time
        })
        
        return ds_merged
    
    except Exception as e:
        print(f"⚠️ Error processing {grib_path}: {e}")
        return None

# Function to process a date folder in parallel
def process_date_folder(date_folder):
    date_path = os.path.join(folderspath, date_folder)
    grib_files = sorted([os.path.join(date_path, f) for f in os.listdir(date_path) if f.endswith(".grib2")])
    
    with ProcessPoolExecutor() as executor:
        ds_list = list(filter(None, executor.map(process_grib_file, grib_files)))
    
    if ds_list:
        try:
            return xr.concat(ds_list, dim="time")
        except ValueError as e:
            print(f"⚠️ Skipping {date_folder} due to concatenation error: {e}")
            return None
    return None

# Process all date folders in parallel
with ProcessPoolExecutor() as executor:
    all_ds_list = list(filter(None, executor.map(process_date_folder, date_folders)))

# Merge all datasets across all days
if all_ds_list:
    final_ds = xr.concat(all_ds_list, dim="time")
    print("✅ Successfully merged all datasets!")
    
    # --- Save raw NetCDF before CRS processing ---
    raw_output_path = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2023_raw.nc"
    final_ds.to_netcdf(raw_output_path)
    print(f"📁 Saved raw dataset to {raw_output_path}")
    
    # --- CRS Check and Reprojection ---
    print(f"Original CRS: {final_ds.rio.crs}")
    
    # Assign CRS if not set (assuming WGS84 lat/lon)
    final_ds.rio.write_crs("EPSG:4326", inplace=True)
    print("Assigned CRS: EPSG:4326")
    
    # Drop old integer x/y coords to avoid rename conflict
    final_ds = final_ds.drop_vars(['x', 'y'], errors='ignore')
    
    # Rename dimensions and set spatial dims
    final_ds = final_ds.rename({'longitude': 'x', 'latitude': 'y'})
    final_ds = final_ds.rio.set_spatial_dims(x_dim='x', y_dim='y')
    
    # Extract 1D x and y coordinates from the 2D grid
    x_1d = final_ds['x'].isel(y=0)  # same x values across rows
    y_1d = final_ds['y'].isel(x=0)  # same y values down columns
    
    # Assign 1D coords
    final_ds = final_ds.assign_coords(x=x_1d, y=y_1d)
    
    # Drop old 2D coords
    final_ds = final_ds.drop_vars(['x', 'y'])
    final_ds = final_ds.set_coords([])
    final_ds = final_ds.assign_coords({'x': x_1d, 'y': y_1d})
    
    # Reproject to UTM
    ds_utm = final_ds.rio.reproject("EPSG:32608")
    print("Reprojected to UTM (EPSG:32608)")
    
    # Save the reprojected dataset
    utm_output_path = "/hdd/snow_hydrology/hrrrak/large_juneau_domain/netcdf/hrrrak_merged_grib_f567_WY2023_utm.nc"
    ds_utm.to_netcdf(utm_output_path)
    print(f"📁 Saved reprojected dataset to {utm_output_path}")
else:
    print("⚠️ No valid datasets found.")

⚠️ Error processing /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023/hrrr.20221005/hrrr.t03z.wrfsfcf06.ak.grib2: cannot rename 't' because it is not a variable or dimension in this dataset⚠️ Error processing /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023/hrrr.20221005/hrrr.t03z.wrfsfcf07.ak.grib2: cannot rename 't' because it is not a variable or dimension in this dataset

⚠️ Error processing /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023/hrrr.20221005/hrrr.t06z.wrfsfcf07.ak.grib2: cannot rename 't' because it is not a variable or dimension in this dataset
⚠️ Error processing /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023/hrrr.20221005/hrrr.t00z.wrfsfcf05.ak.grib2: cannot rename 't' because it is not a variable or dimension in this dataset⚠️ Error processing /hdd/snow_hydrology/hrrrak/large_juneau_domain/f567/WY2023/hrrr.20221005/hrrr.t00z.wrfsfcf06.ak.grib2: cannot rename 't' because it is not a variable or dimension in this dataset

⚠️ Er

Process ForkProcess-24:38:
Process ForkProcess-23:43:
Process ForkProcess-24:36:
Process ForkProcess-24:35:
Process ForkProcess-24:31:
Process ForkProcess-24:45:
Process ForkProcess-24:46:
Process ForkProcess-24:32:
Process ForkProcess-24:34:
Process ForkProcess-23:47:
Process ForkProcess-24:37:
Process ForkProcess-24:48:
Process ForkProcess-24:44:
Process ForkProcess-24:28:
Process ForkProcess-23:45:
Process ForkProcess-24:47:
Process ForkProcess-23:46:
Process ForkProcess-24:40:
Process ForkProcess-24:41:
Process ForkProcess-24:39:
Process ForkProcess-24:33:
Process ForkProcess-23:48:
Process ForkProcess-23:41:
Process ForkProcess-24:30:
Traceback (most recent call last):
Process ForkProcess-24:26:
Process ForkProcess-23:39:
Process ForkProcess-24:29:
Process ForkProcess-24:42:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkProcess-23:38:
Process ForkProcess-23:42:
Traceback (most

KeyboardInterrupt: 